<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
RAG-Agent
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Hier wird das Evidence Tool zu einer Fähigkeit unter mehreren — der PMO-Assistent **plant** jetzt selbst, ob eine Frage RAG braucht, reines Modellwissen ausreicht oder ein anderes Tool gefragt ist (siehe Kapitel 4). Der Grenzfall-Test in Kapitel 5 prüft zusätzlich, ob der Agent bei fehlender Evidenz die Evidenzgrenze benennt, zum Beispiel mit "Nicht im Korpus". Der Wortlaut darf dabei abweichen; ob der Standardwortlaut vorkommt, weist die Prüfung separat aus. Die vollständigen Antworten müssen anschließend gegen die Quellen geprüft werden.

**Praxis-Transfer:** Mara Vogt nutzt den RAG-Agenten, wenn der Agent selbst entscheiden muss, wann eine Dokumentensuche für ihr Briefing erforderlich ist.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith-Variablen VOR allen LangChain-Imports setzen.
# Globales Tracing bleibt aus; ausgewählte Testläufe aktivieren es gezielt.
import os
os.environ["LANGSMITH_TRACING"] = "false"
from langsmith import tracing_context
os.environ["LANGSMITH_PROJECT"] = "m12_rag_agent"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    WORKER,
    CODING,
    JUDGE,
    PLANNER,
    WORKER_PREMIUM,
    FRONTIER,
    IMAGE_GENERATION,
    TRANSCRIPTION,
    TRANSCRIPTION_SEGMENTS,
    EMBEDDINGS,
)

In [ ]:
#@title 📂 PMO-Korpus laden { display-mode: "form" }
import shutil
from genai_lib.utilities import copy_from_github

CORPUS_SOURCE = "ralf-42/Agenten/02_daten/01_text/korpus_pmo_assistent"
CORPUS_PATTERN = "*"
CORPUS_TARGET = "/content/files"

shutil.rmtree(CORPUS_TARGET, ignore_errors=True)
copy_from_github(
    source=CORPUS_SOURCE,
    target=CORPUS_TARGET,
    mask=CORPUS_PATTERN,
)


# 1 | Übersicht
---

In *RAG Grundlagen* wurde eine **RAG-Chain** gebaut: deterministisch, mit linearem Ablauf.  
Jetzt folgt der nächste Schritt: **RAG als Tool für einen Agenten**.

**Von der Chain zum Agent**

| Merkmal | RAG-Chain | RAG-Agent |
|---------|----------------|----------------|
| Ablauf | Immer: Retrieve → Generate | Nur bei Bedarf: Agent entscheidet |
| RAG-Nutzung | Immer aktiv | Tool, das der Agent auswählt |
| Weiteres Wissen | Nur Vektordatenbank | Modellwissen + RAG + weitere Tools |
| Flexibilität | Niedrig | Hoch |
| Komplexität | Gering | Mittel |

**Was entsteht**

Ein Agenten-System mit **RAG als einem von mehreren Tools**:
- `search_knowledge_base` – RAG-Tool für belegbare Fragen zum PMO-Korpus
- `explain_project_term` – Direktes Modellwissen für allgemeine Erklärungen
- `estimate_token_cost` – Einfaches Berechnungstool für Kontextkosten


<p><font color='darkblue' size="4">
 <b>✨ Viz</b>
</font></p>

- [RAG-Pipeline](https://ralf-42.github.io/Agenten/visualisierungen/rag_prozess_visualisierung/)
- [KI-Agent](https://ralf-42.github.io/Agenten/visualisierungen/ki_agenten/)


# 2 | RAG als Tool
---

Ein **RAG-Tool** ist eine normale LangChain-Tool-Funktion, die intern ChromaDB abfragt. Der Agent entscheidet, ob diese Quelle nötig ist.


Der Index-Aufbau ist hier Voraussetzung, nicht das Lernziel. Der Fokus liegt auf Tool-Definition, Tool-Wahl und Quellen-Grenzen des Agenten.



<p><font color='black' size="5">
Design-Prinzipien für RAG-Tools
</font></p>


| Prinzip | Beschreibung |
|---------|-------------|
| **Klarer Name** | Name sagt dem Agenten, wann das Tool zu nutzen ist |
| **Präzise Beschreibung** | Docstring erklärt genau, welche Fragen das Tool beantwortet |
| **Fehlerbehandlung** | Gibt lesbaren Text zurück, kein `raise` |
| **Formatierte Ausgabe** | Gibt Quellen + Antwort zurück |

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser

from genai_lib.pmo_rag import (
    format_docs,
    get_pmo_vectorstore,
    index_pmo_corpus,
)

In [ ]:
# LLM
llm = init_chat_model(WORKER)

📦 **Hinweis zu `genai_lib.pmo_rag`:** Mehrformat-Laden, Chunking und Chroma-Aufbau (mit Batching) kommen aus dem wiederverwendbaren Modul `genai_lib.pmo_rag` — hier über `format_docs`, `get_pmo_vectorstore` und `index_pmo_corpus` (installierbar via `git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul`). Das Tool `search_knowledge_base` entsteht dagegen bewusst im Notebook (Kapitel 3), weil die Tool-Definition hier Lernziel ist; als fertige Variante enthält das Modul zusätzlich `make_suche_wissensdatenbank_tool()`. Wie Laden, Chunking und Indexierung im Detail funktionieren, zeigt RAG Grundlagen. Die Collection liegt dauerhaft auf Google Drive (`Agenten/02_daten/05_sonstiges/chroma_pmo`) und wird bei Folgeläufen nicht neu indexiert.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Die Funktion prüft Dateibestand und Quellenmetadaten und aktualisiert den
# persistenten Index nur, wenn sich der Korpus geändert hat.
vectorstore = get_pmo_vectorstore()
indexed_chunks = index_pmo_corpus(
    vectorstore,
    CORPUS_TARGET,
    chunk_size=300,
    chunk_overlap=30,
)

if indexed_chunks:
    mprint(f"Index aktualisiert: {indexed_chunks} Chunks neu indexiert")
mprint(f"Collection-Größe: {vectorstore._collection.count()}")


# 3 | RAG-Agent bauen
---



In [ ]:
# Gemeinsame Anleitung für RAG-Tool und abschließende Agentenantwort.
# Keine Referenzfakten vorgeben: Alle Aussagen müssen aus den Quellen stammen.
comparison_instructions = (
    "Bei Entscheidungsvergleichen für jede Entscheidung Inhalt, Geltungsbereich, "
    "Zeitpunkt und Status aus den Quellen nennen und die Unterschiede erklären. "
    "Betriebsmodell und konkrete Anbieterwahl getrennt behandeln. "
    "Fehlt eine Anbieterangabe, darf ein belegtes Betriebsmodell nicht entfallen. "
    "Bei offenen Punkten auch die bereits erledigten Voraussetzungen und die "
    "verantwortliche Rolle nennen, soweit sie belegt sind. "
    "Fehlende Informationen gezielt benennen und keine Details ergänzen. "
    "Vor der Antwort prüfen, ob alle belegten Vergleichspunkte erhalten geblieben sind."
)

# RAG-Bausteine: Retrieval und Formatierung (format_docs aus genai_lib.pmo_rag) bleiben außerhalb der Chain sichtbar
# Sechs Chunks erweitern den Kontext für Vergleiche wie D1/D2; Vollständigkeit bleibt zu prüfen.
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})


rag_prompt = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m11_rag_prompt.md",
    mode="T",
)
parser = StrOutputParser()
rag_chain = rag_prompt | llm | parser

In [ ]:
# Tools definieren
retrieval_minutes = []


@tool
def search_knowledge_base(question: str) -> str:
    """Beantwortet belegbare Fragen mit dem PMO-Korpus.

    Das Argument enthält nur die Nutzerfrage; Formatvorgaben für die Antwort
    ergänzt das Tool selbst, damit die Suche nicht mit Zusatztext läuft.

    Die Verarbeitung bleibt bewusst in drei sichtbaren Schritten:
    Retrieval, Kontextformatierung und Aufruf der reinen RAG-Chain.
    """
    try:
        # 1. Dokumente explizit abrufen
        retrieved = retriever.invoke(question, config={"run_name": "m12_rag_retrieval"})
        sources = []
        for doc in retrieved:
            source = str(doc.metadata.get("source", "?")).replace("\\", "/").rsplit("/", 1)[-1]
            if source not in sources:
                sources.append(source)
        # Chunk-Texte mitprotokollieren: nur so ist später prüfbar, ob ein
        # fehlender Fakt am Retrieval oder an der Antwort lag.
        retrieval_minutes.append({
            "frage": question,
            "quellen": sources,
            "anzahl": len(retrieved),
            "ausschnitte": [doc.page_content for doc in retrieved],
        })
        if not retrieved:
            return "Nicht im Korpus. Keine relevanten Informationen gefunden."

        # 2. Dokumente formatieren
        context_text = format_docs(retrieved)

        # 3. Chain mit dem fertigen Dictionary aufrufen
        answer = rag_chain.invoke(
            {"context": context_text, "question": f"{question}\n\n{comparison_instructions}"},
            config={"run_name": "m12_rag_chain_antwort"},
        )
        return answer
    except Exception:
        # Systemgrenze: Details nur im Log, nicht als Tool-Antwort.
        import logging
        logging.getLogger(__name__).exception("Datenbanksuche fehlgeschlagen")
        return "Die Datenbanksuche ist fehlgeschlagen. Bitte später erneut versuchen."

In [ ]:
@tool
def explain_project_term(term: str) -> str:
    """Erklärt einen allgemeinen Projektmanagement-Begriff (zum Beispiel Steuerkreis, Mitigation, Action Item) kurz und verständlich.
    Geeignet für Begriffsfragen ohne zwingenden Korpusbezug.
    """
    prompt = f"Erkläre den Begriff '{term}' in 2-3 Sätzen für Teilnehmende eines Projekt-Steuerkreises."
    return llm.invoke(prompt, config={"run_name": "m12_begriff_erklaerung"}).text

In [ ]:
# Beispielpreis, Stand 11.2026 — keine Preisauskunft, nur Größenordnung für die Demo.
PREIS_PRO_1000_TOKENS_USD = 0.005


@tool
def estimate_token_cost(text: str, model: str = WORKER) -> str:
    """Schätzt Tokenzahl und grobe Eingabekosten für einen Text."""
    characters = len(text)
    tokens_estimated = characters // 3
    cost = (tokens_estimated / 1000) * PREIS_PRO_1000_TOKENS_USD
    return (
        f"Text: {characters} Zeichen\n"
        f"Geschätzte Tokens: ~{tokens_estimated} (Faustregel: 3 Zeichen/Token)\n"
        f"Geschätzte Kosten für {model}: ~${cost:.6f} "
        f"(Beispielpreis {PREIS_PRO_1000_TOKENS_USD} USD je 1000 Tokens, Stand 11.2026)"
    )

In [ ]:
tools = [search_knowledge_base, explain_project_term, estimate_token_cost]

In [ ]:
# RAG-Agent erstellen
system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m12_rag_agent_system_prompt.md", mode="S")

# Ergänzung gilt sofort, auch wenn der Basis-Prompt von GitHub geladen wird.
system_prompt = f"{system_prompt}\n\n{comparison_instructions}"

rag_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    debug=False
)

# 4 | Wann RAG, wann eigenes Wissen?
---



Der Agent entscheidet selbst, welches Tool er nutzt – aber wie trifft er diese Entscheidung?

**Entscheidungslogik**

Der Agent wählt das Tool basierend auf **drei Faktoren**:

1. **Tool-Name** – Sprechende Namen helfen: `search_knowledge_base` vs. `explain_project_term`
2. **Docstring** – Präzise Beschreibung, wann das Tool einzusetzen ist
3. **Kontext der Frage** – Spezifisch (→ RAG) vs. allgemein (→ eigenes Wissen)

In [ ]:
#@markdown   <p><font size="4" color='green'>  RAG-Tool</font> </br></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TD
    FRAGE["❓ Nutzerfrage"]

    FRAGE --> SPEZ{{"Briefing-Frage mit Quellenbedarf?
(Entscheidungen,
Risiken, Aufgaben)"}}
    SPEZ -->|"Ja"| RAG["📚 search_knowledge_base()
ChromaDB Retrieval"]
    SPEZ -->|"Nein"| ALLG{{"Allgemeines
Konzept?"}}

    ALLG -->|"Ja"| LLM["🧠 Direkte Erklärung
Optional: explain_project_term()"]
    ALLG -->|"Nein"| CALC{{"Berechnung
nötig?"}}

    CALC -->|"Ja"| TOK["🔢 estimate_token_cost()
Faustregel"]
    CALC -->|"Nein"| DIRECT["💬 Direkte Antwort"]

    RAG --> ANS["Antwort"]
    LLM --> ANS
    TOK --> ANS
    DIRECT --> ANS

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class FRAGE input;
    class SPEZ,ALLG,CALC metric;
    class ANS output;
    class RAG,TOK target;
'''

mermaid(diagram, width=1050)

In [ ]:
# Test: Agent zeigt unterschiedliche Tool-Wahl
test_questions = [
    "Wie unterscheiden sich die Entscheidungen D1 und D2 zur Vektordatenbank?",
    "Welche Aufgabe ist laut Follow-up-Tracker noch offen?",
    "Was bedeutet der Begriff Steuerkreis in einem Projekt?",
    "Schätze die Tokenzahl für diesen kurzen Beispieltext: Risiko R2 (Datenschutzfreigabe) wurde am 28.04.2026 von offen auf mitigiert gesetzt.",
]

In [ ]:
# LangSmith Konfiguration — run_name setzt der jeweilige Aufruf
run_cfg = {
    "tags": ["rag_agent", "pmo_briefing_korpus"],
    "metadata": {"notebook": "rag_agent", "version": "1.0"},
}

In [ ]:
# Hilfsfunktion: verwendete Tool-Namen aus dem Agenten-Ergebnis lesen
def tool_names(result) -> list[str]:
    """Liest die Tool-Namen aus den ToolMessages, ersatzweise aus AIMessage.tool_calls."""
    messages = result["messages"]
    names = [
        m.name
        for m in messages
        if getattr(m, "type", None) == "tool" and getattr(m, "name", None)
    ]
    if not names:
        names = [
            call["name"]
            for m in messages
            for call in getattr(m, "tool_calls", [])
            if call.get("name")
        ]
    return names

In [ ]:
# Aufruf Agenten
from langsmith import tracing_context

tool_selection_results = []
with tracing_context(enabled=True):
    for _i, (question) in enumerate(test_questions, 1):
        mprint(f"Frage: {question}")
        retrieval_start = len(retrieval_minutes)
        result = rag_agent.invoke(
            {"messages": [HumanMessage(content=question)]},
            config={**run_cfg, "run_name": f"m12_kap4_rag_agent_ergebnis_{_i}"},
        )
        new_retrievals = retrieval_minutes[retrieval_start:]
        for retrieval in new_retrievals:
            sources_text = ", ".join(retrieval["quellen"]) or "keine"
            mprint(f"Suchfrage: {retrieval['frage']}")
            mprint(f"Treffer: {retrieval['anzahl']} Chunks aus {len(retrieval['quellen'])} Quelle(n): {sources_text}")
            for nummer, ausschnitt in enumerate(retrieval["ausschnitte"], 1):
                # Rohtext bewusst mit print: Markdown würde Tabellen im Chunk umformatieren.
                print(f"  Chunk {nummer}: {ausschnitt[:200].replace(chr(10), ' ')}")

        last_answer = result["messages"][-1].text
        used_tools = tool_names(result)
        tool_selection_results.append({
            "frage": question,
            "tool_namen": used_tools,
            "tool_calls": len(used_tools),
            "antwort": last_answer,
        })
        mprint(f"Tool-Wahl: {used_tools or ['direkte Antwort']}")
        mprint(f"Antwort: {last_answer}")
        mprint("-" * 60)

# 5 | End-to-End Testing
---

Ein gutes RAG-System braucht systematische Tests.

| Testtyp | Was wird geprüft? | Methode |
|---------|-------------------|---------|
| **Retrieval-Test** | Werden relevante Dokumente gefunden? | Chunks und Quellen prüfen |
| **Antwortqualität** | Sind Sachvergleich und Status vollständig und belegt? | Vollständige Antwort mit Quellen vergleichen |
| **Tool-Wahl** | Nutzt der Agent die Dokumentensuche? | Tool-Namen und Trace prüfen |
| **Grenzfall** | Benennt der Agent die fehlende Evidenz? | Bekannte Ablehnungsformulierungen prüfen; Standardwortlaut separat ausweisen |

**Automatische Plausibilitätsprüfung:** `CHECK_OK` bedeutet, dass die erwarteten Begriffe im Antwortinhalt, die RAG-Tool-Nutzung und bei beantwortbaren Fragen ein tatsächlich abgerufener Quellenname gefunden wurden. Das beweist weder die korrekte Zuordnung von Aussagen zu D1/D2 noch ihre Belegbarkeit. Die vollständigen Antworten und die zitierten Passagen müssen manuell geprüft werden. `CHECK_FAIL` zeigt fehlende Kriterien.

**Inhalt und Quellen:** Die Begriffsprüfung entfernt gekennzeichnete Quellenzeilen, Dateinamen, URLs und Markdown-Links. So kann „Cloud“ in einem Dateinamen keine fehlende Aussage ersetzen. Die Quellenprüfung sucht separat im Original nach tatsächlich abgerufenen Dateinamen. Diese einfache Trennung ist konservativ: Inhalt in Links oder ungewöhnlich formatierte Quellenhinweise muss gegebenenfalls manuell geprüft werden.

**Entscheidungsvergleiche:** RAG-Tool und Agent erhalten dieselbe allgemeine Anleitung: Inhalt, Geltungsbereich, Zeitpunkt und Status vergleichen sowie Betriebsmodell und Anbieterwahl unterscheiden. Referenzfakten werden nicht in den Prompt eingebaut. Eine korrekt benannte Informationslücke zu einem Zusatzaspekt macht einen sonst vollständigen Vergleich nicht automatisch falsch.

**Wortlaut und Ablehnung:** Der Out-of-Corpus-Test akzeptiert neben „Nicht im Korpus“ bekannte Wortstellungen wie „im PMO-Korpus nicht belegt“ und „nicht im PMO-Korpus enthalten“. Schreibweise, Markdown und Leerzeichen werden dafür vereinheitlicht. Ob der Standardwortlaut vorkommt, wird separat ausgewiesen und entscheidet nicht allein über den Status. Eine bekannte Formulierung ist ein Hinweis auf eine Ablehnung, kein semantischer Beweis: Unbekannte Formulierungen, Verneinungen und unbelegte Zusätze müssen manuell geprüft werden.

Ein größeres `k` kann den D1/D2-Vergleich verbessern, garantiert aber keine passenden Treffer. Fehlen Informationen, die Suchfragen und Chunks im Trace prüfen; die Antwort nicht durch vorgegebene Fakten ersetzen.


In [ ]:
# End-to-End Test: zwei Korpusfragen und eine bewusst unbeantwortbare Frage
# "erwartungen" prüft den Antworttext, "retrieval_erwartungen" die abgerufenen Chunks.
test_scenarios = [
    {
        "frage": "Wie unterscheiden sich die Entscheidungen D1 und D2 zur Vektordatenbank?",
        "erwartungen": {
            "D1 genannt": ["d1"],
            "D2 genannt": ["d2"],
            "ChromaDB erwähnt": ["chromadb"],
            "D1: Prototyp": ["prototyp"],
            "D2: verwaltete Cloud-Vektordatenbank": ["cloud"],
            "D2: verwalteter Betrieb": ["verwaltet", "managed"],
            "D2: produktiver Einsatz": ["produktiv", "produktion"],
        },
        # Der Antworttext kann "chromadb" auch in einer Einschränkung enthalten.
        # Deshalb separat prüfen, ob die Zuordnung überhaupt abgerufen wurde.
        "retrieval_erwartungen": [["d1", "chromadb"]],
        "quellenpflicht": True,
    },
    {
        "frage": "Welche Aufgabe ist laut Follow-up-Tracker noch offen?",
        "erwartungen": {
            "Rechte": ["rechte"],
            "Rollen": ["rollen"],
            "Status offen": ["offen", "vertagt"],
            "Verantwortliche Rolle genannt": ["mara vogt"],
        },
        "retrieval_erwartungen": [["rollensteuerung"]],
        "quellenpflicht": True,
    },
    {
        "frage": "Welche Umsatzzahlen erzielte die Nordlicht Software GmbH im letzten Geschäftsjahr laut PMO-Korpus?",
        "erwartungen": {},
        "retrieval_erwartungen": [],
        "ablehnung_erwartet": True,
        "quellenpflicht": False,
    },
]

In [ ]:
# LangSmith Konfiguration — run_name setzt der jeweilige Aufruf
run_cfg = {"tags": ["test", "pmo_briefing_korpus"], "metadata": {"notebook": "rag_agent", "version": "1.0"}}

In [ ]:
import re
import unicodedata


def normalize_check_text(text):
    """Vereinheitlicht Schreibweise und Markdown für die einfachen Textchecks."""
    text = unicodedata.normalize("NFKC", text).casefold()
    text = re.sub(r"[`*_~]", "", text)
    text = re.sub(r"[‐‑‒–—−]", "-", text)
    return re.sub(r"\s+", " ", text).strip()


def answer_content_for_checks(answer, source_names):
    """Entfernt Quellenhinweise; Zitate werden separat am Original geprüft."""
    text = re.sub(r"\[[^\]]*\]\([^)]+\)", "", answer)
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"(?im)[*_`]*(?:quellen?|belege|referenzen)[*_`]*\s*:[^\n]*", "", text)
    for name in source_names:
        text = re.sub(re.escape(name), "", text, flags=re.IGNORECASE)
    text = re.sub(r"(?i)\b[\w.-]+\.(?:pdf|md|txt|csv|xlsx?|docx?|html?)\b", "", text)
    return normalize_check_text(text)


def find_refusal_phrase(text):
    """Erkennt bekannte Ablehnungsformen, ohne ihre sachliche Gültigkeit zu beurteilen."""
    corpus = r"(?:pmo[- ]?)?korpus"
    qualifier = r"(?:belegt|enthalten|dokumentiert|vorhanden|verfügbar)"
    patterns = [
        rf"\bnicht im {corpus}(?: {qualifier})?\b",
        rf"\bim {corpus} nicht {qualifier}\b",
    ]
    return next((match.group() for pattern in patterns
                 if (match := re.search(pattern, text))), None)


In [ ]:
# Inhalt, abgerufene Chunks, Quellenangaben und Ablehnungswortlaut getrennt prüfen.
# Floskeln, die auf eine ungeklärte Zuordnung hindeuten — kein Fehler, aber ein Prüfanlass.
UNSICHERHEITSFLOSKELN = ["nicht eindeutig", "nicht zuzuordnen", "nicht zuordenbar"]


def check_rag_answer(answer, used_tools, retrievals, scenario):
    retrieved_sources = sorted({
        name for retrieval in retrievals for name in retrieval["quellen"]
        if name and name != "?"
    })
    chunks = [
        normalize_check_text(ausschnitt)
        for retrieval in retrievals for ausschnitt in retrieval.get("ausschnitte", [])
    ]
    content = answer_content_for_checks(answer, retrieved_sources)
    missing = [
        name for name, variants in scenario["erwartungen"].items()
        if not any(normalize_check_text(value) in content for value in variants)
    ]
    # Retrieval-Test: Stand der erwartete Fakt überhaupt in einem abgerufenen Chunk?
    missing_in_chunks = [
        " + ".join(gruppe)
        for gruppe in scenario.get("retrieval_erwartungen", [])
        if not any(
            all(normalize_check_text(begriff) in chunk for begriff in gruppe)
            for chunk in chunks
        )
    ]
    cited_sources = [name for name in retrieved_sources if name.casefold() in answer.casefold()]
    refusal_expected = scenario.get("ablehnung_erwartet", False)
    refusal_phrase = find_refusal_phrase(content)
    if refusal_expected and not refusal_phrase:
        missing.append("Evidenzgrenze genannt: keine bekannte Ablehnungsformulierung")
    if "search_knowledge_base" not in used_tools:
        missing.append("RAG-Tool verwendet")
    if scenario["quellenpflicht"] and not cited_sources:
        missing.append("Abgerufene Quelle in Antwort genannt")
    hedges = [] if refusal_expected else [f for f in UNSICHERHEITSFLOSKELN if f in content]
    # Bei Korpusfragen zählen die geforderten Inhalte: Eine ergänzende Aussage
    # zur unbekannten Anbieterwahl macht einen vollständigen Vergleich nicht falsch.
    return {
        "status": "CHECK_FAIL" if (missing or missing_in_chunks) else "CHECK_OK",
        "fehlende_erwartungen": missing,
        "fehlende_chunk_fakten": missing_in_chunks,
        "unsicherheitsfloskeln": hedges,
        "ablehnungswortlaut": bool(re.search(r"\bnicht im korpus\b", content)) if refusal_expected else None,
        "erkannte_ablehnungsformulierung": refusal_phrase if refusal_expected else None,
        "zitierte_quellen": cited_sources,
        "abgerufene_quellen": retrieved_sources,
    }

In [ ]:
# End-to-End Test ausführen und vollständige Antworten zur Prüfung anzeigen
mprint("=== End-to-End Plausibilitätsprüfung ===")
e2e_results = []
with tracing_context(enabled=True):
    for i, scenario in enumerate(test_scenarios, 1):
        question = scenario["frage"]
        mprint(f"Test {i}: {question}")
        retrieval_start = len(retrieval_minutes)
        result = rag_agent.invoke(
            {"messages": [HumanMessage(content=question)]},
            config={**run_cfg, "run_name": f"m12_kap5_rag_agent_ergebnis_{i}"},
        )
        new_retrievals = retrieval_minutes[retrieval_start:]
        for retrieval in new_retrievals:
            sources_text = ", ".join(retrieval["quellen"]) or "keine"
            mprint(f"  Suchfrage: {retrieval['frage']}")
            mprint(f"  Treffer: {retrieval['anzahl']} Chunks aus {len(retrieval['quellen'])} Quelle(n): {sources_text}")
            for nummer, ausschnitt in enumerate(retrieval["ausschnitte"], 1):
                # Rohtext bewusst mit print: Markdown würde Tabellen im Chunk umformatieren.
                print(f"    Chunk {nummer}: {ausschnitt[:200].replace(chr(10), ' ')}")
        answer = result["messages"][-1].text
        used_tools = tool_names(result)
        check = check_rag_answer(answer, used_tools, new_retrievals, scenario)
        e2e_results.append({"frage": question, "antwort": answer, "tool_namen": used_tools, **check})
        mprint(f"  Tool-Wahl: {used_tools or ['direkte Antwort']}")
        mprint(f"  Antwort:\n{answer}")
        mprint(f"  Zitierte Trefferquellen: {', '.join(check['zitierte_quellen']) or 'keine'}")
        if not scenario["quellenpflicht"]:
            mprint(f"  Standardwortlaut 'Nicht im Korpus': {'ja' if check['ablehnungswortlaut'] else 'nein'}")
            mprint(f"  Erkannte Ablehnungsformulierung: {check['erkannte_ablehnungsformulierung'] or 'keine bekannte Variante'}")
            mprint("  Grenzfall: keine Quelle erforderlich; Antwort auf erfundene Angaben prüfen.")
        mprint(f"  Fehlende Kriterien: {', '.join(check['fehlende_erwartungen']) or 'keine'}")
        mprint(f"  Fakten ohne Chunk-Beleg: {', '.join(check['fehlende_chunk_fakten']) or 'keine'}")
        if check["unsicherheitsfloskeln"]:
            mprint(f"  Unsicherheitsfloskeln in der Antwort: {', '.join(check['unsicherheitsfloskeln'])}")
        mprint(f"  Status: {check['status']} — fachliche Quellenprüfung erforderlich.")
        mprint("-" * 40)

**Trace prüfen**

Die Läufe dieses Kapitels liegen im LangSmith-Projekt aus `LANGSMITH_PROJECT`. Der Trace zeigt, mit welchem Argument der Agent die Dokumentensuche aufgerufen hat und welche Chunks dabei zurückkamen. Genau das entscheidet, ob ein fehlender Fakt am Retrieval oder an der Antwort lag.

In [ ]:
# Trace des Kapitels: Tool-Wahl, Tool-Argumente und Retrieval-Runs nachvollziehen
show_trace(os.environ["LANGSMITH_PROJECT"], limit=6, show_steps=True)

In [ ]:
# Diagnose: Wirkt der Zusatztext im Tool-Argument auf das Retrieval?
# Vergleicht die reine Nutzerfrage mit derselben Frage plus Formatvorgaben.
rohe_frage = "Wie unterscheiden sich die Entscheidungen D1 und D2 zur Vektordatenbank?"
varianten = {
    "Nur Frage": rohe_frage,
    "Frage + Formatvorgaben": f"{rohe_frage}\n\n{comparison_instructions}",
}

for titel, suchtext in varianten.items():
    treffer = retriever.invoke(suchtext, config={"run_name": "m12_kap5_retrieval_diagnose"})
    quellen = []
    for doc in treffer:
        name = str(doc.metadata.get("source", "?")).replace("\\", "/").rsplit("/", 1)[-1]
        if name not in quellen:
            quellen.append(name)
    mit_fakt = sum(
        1 for doc in treffer
        if "d1" in doc.page_content.casefold() and "chromadb" in doc.page_content.casefold()
    )
    mprint(f"{titel}: {len(treffer)} Chunks, davon {mit_fakt} mit D1 und ChromaDB zusammen")
    mprint(f"Quellen: {', '.join(quellen) or 'keine'}")

**Zusammenfassung: Wann was?**

| Kriterium | RAG-Chain | RAG-Agent |
|-----------|-----------|-----------|
| Einfache Q&A | Ideal | Overhead |
| Gemischte Fragen | Nur RAG | Flexibel |
| Mehrere Tools | Komplex | Geeignet |
| Nachvollziehbarkeit | Deterministisch | Trace nötig |
| Einstieg | Empfohlen | Nachgelagert |

**Empfehlung:** Mit RAG-Chain starten, bei Bedarf zum RAG-Agenten ausbauen.


# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**
- PMO-Korpus als RAG-Tool für den Agenten verwenden.
- search_knowledge_base mit mindestens einer Frage aus dem Eval-Set testen.
- Ergebnis mit mindestens einer Quelle und einem kurzen Ausschnitt anzeigen.
- Die Antwort zusätzlich in grundlagen_ergebnis speichern.
- Fertigen Agenten in my_rag_agent speichern.


In [ ]:
# Grundlagen: rag_agent aus Kapitel 3 für die Aufgabe verwenden
my_rag_agent = rag_agent

# 1. Frage aus dem Eval-Set wählen (z.B. "Wie unterscheiden sich die Entscheidungen D1 und D2 zur Vektordatenbank?")
# 2. my_rag_agent.invoke() mit der Frage aufrufen (config mit run_name/tags)
# 3. Vollständige Antwort in grundlagen_ergebnis speichern
# 4. Prüfen, ob Quelle: oder ein PDF-Dateiname enthalten ist


**Aufbau**
- Agenten mit drei Tools prüfen: Briefing-Dokumentensuche, Begriffserklärung und Token-Schätzung.
- Drei Fragen testen: eine Briefing-Frage mit Quellenbedarf, eine allgemeine Konzeptfrage, eine Berechnungsfrage.
- Die tatsächlich verwendeten Tool-Namen und Antwortauszüge dokumentieren.


In [ ]:
# Aufbau: drei Fragetypen mit demselben Agenten testen
# Startpunkt: my_rag_agent aus Grundlagen

# 1. Drei Fragen definieren: Briefing-Frage mit Quellenbedarf, allgemeine Konzeptfrage, Berechnungsfrage
# 2. Für jede Frage my_rag_agent.invoke() aufrufen
# 3. Tool-Namen mit tool_names(result) (Kapitel 4) auslesen
# 4. Ergebnisse in toolwahl_ergebnisse sammeln (frage, tool_namen, tool_calls, antwort)


**Vertiefung**
- Einen LangSmith-Test mit tracing_context, run_name und tags ergänzen.
- Tool-Wahl für Briefing-Frage mit Quellenbedarf, Konzeptfrage und Out-of-Corpus-Frage vergleichen.
- Prüfen, ob RAG-Antworten einen Quellenhinweis enthalten.
- Fälle mit unerwarteter Tool-Wahl dokumentieren.
- Beobachtung in _tool_choice_observation speichern.


In [ ]:
# Vertiefung: LangSmith-Analyse der Tool-Wahl
# Startpunkt: my_rag_agent, toolwahl_ergebnisse aus Aufbau

from langsmith import tracing_context

_tool_choice_observation = ""
# 1. Eine Grenzfall-Frage außerhalb des Korpus formulieren.
# 2. Projekt-, Begriffs- und Grenzfall-Frage in einem tracing_context-Kontext
#    mit run_name und tags aufrufen.
# 3. Tool-Namen und Quellenhinweise vergleichen.
# 4. Mindestens eine beobachtete Grenze in _tool_choice_observation festhalten.


**Praxis-Transfer-Aufgabe: PMO-Assistent**

RAG als Tool: Agent entscheidet, wann Quellen für Meeting- oder Projektkontext nötig sind.

1. Welche reale Arbeitsfrage löst **dieser** Notebook-Baustein im PMO-Assistent?
2. Welche Eingabedaten oder Dokumente braucht er?
3. Welche Ausgabe sollte er liefern?
4. Welche Risiken oder Grenzen bleiben?
5. Wann wäre Human Review nötig?
6. Ist der passende Lösungsweg hier Prompt, strukturierte Ausgabe, RAG, Tool, Agent oder Workflow?

# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [RAG-Konzepte](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/rag-konzepte.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)
- [Context Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/context-engineering.html)
- [Einsteiger ChromaDB](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-chromadb.html)
- [Evaluation & Observability](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/evaluation-observability.html)
